# DA2-MODEL-01 — Chuẩn bị dữ liệu train/test cho tín hiệu Buy/Sell

**Input:** `sv3/DA2-DATA-06/processed_data/bitcoin.parquet`  
**Output:** `X_train, X_test, y_train, y_test, scaler` (dùng lại ở MODEL-02, 03, 04)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import pickle, os

# Load feature table từ SV3
df = pd.read_parquet('/home/jovyan/work/sv3/DA2-DATA-06/processed_data/bitcoin.parquet')
df = df.sort_values('timestamp').reset_index(drop=True)

print('Shape:', df.shape)
print('Columns:', list(df.columns))
df.head()

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print('\nPhân bố label:')
print(df['label'].value_counts())
print(f'  Buy (1): {(df["label"]==1).sum()} ({(df["label"]==1).mean()*100:.1f}%)')
print(f'  Sell(0): {(df["label"]==0).sum()} ({(df["label"]==0).mean()*100:.1f}%)')

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 3))
plt.scatter(df.index, df['close'], c=df['label'], cmap='RdYlGn', s=5, alpha=0.7)
plt.colorbar(label='0=Sell / 1=Buy')
plt.title('Giá Bitcoin theo thời gian (màu = Buy/Sell label)')
plt.xlabel('Index')
plt.ylabel('Close Price (USD)')
plt.tight_layout()
plt.show()

In [ ]:
# Không shuffle — chia theo thứ tự thời gian
FEATURES = ['MA10', 'MA60', 'ROC', 'MOM', 'RSI', 'stoch_k', 'stoch_d']
TARGET   = 'label'

X = df[FEATURES]
y = df[TARGET]

split = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

print(f'Tổng mẫu : {len(df)}')
print(f'Train     : {len(X_train)} ({len(X_train)/len(df)*100:.0f}%)')
print(f'Test      : {len(X_test)}  ({len(X_test)/len(df)*100:.0f}%)')
print(f'Features  : {FEATURES}')

In [ ]:
# Chuẩn hóa — fit trên train, transform cả hai
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print('StandardScaler fit xong.')
print('Mean:', scaler.mean_.round(4))
print('Std :', scaler.scale_.round(4))

# Lưu để dùng ở các notebook sau
os.makedirs('/home/jovyan/work/sv4/shared', exist_ok=True)
with open('/home/jovyan/work/sv4/shared/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
np.save('/home/jovyan/work/sv4/shared/X_train_sc.npy', X_train_sc)
np.save('/home/jovyan/work/sv4/shared/X_test_sc.npy',  X_test_sc)
np.save('/home/jovyan/work/sv4/shared/y_train.npy',    y_train.values)
np.save('/home/jovyan/work/sv4/shared/y_test.npy',     y_test.values)

print('Đã lưu vào sv4/shared/ để dùng ở MODEL-02, 03, 04, EDA-01')